In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
if os.path.exists("heart_train.csv") and os.path.exists("heart_test.csv"):
    train_df = pd.read_csv("heart_train.csv")
    test_df = pd.read_csv("heart_test.csv")
else:
    df = pd.read_csv("heart_cleaned.csv")
    rng = np.random.default_rng(42)

    train_indices = []
    test_indices = []

    for label in df["target"].unique():
        indices = np.where(df["target"].to_numpy() == label)[0]
        rng.shuffle(indices)

        split_point = int(round(len(indices) * 0.8))
        train_indices.extend(indices[:split_point])
        test_indices.extend(indices[split_point:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    train_df = df.iloc[train_indices].copy()
    test_df = df.iloc[test_indices].copy()



In [4]:
X_train = train_df.drop(columns=["target"]).to_numpy(dtype=float)
y_train = train_df["target"].to_numpy(dtype=float)

X_test = test_df.drop(columns=["target"]).to_numpy(dtype=float)
y_test = test_df["target"].to_numpy(dtype=float)

if set(np.unique(y_train)) != {0.0, 1.0}:
    raise ValueError("This implementation requires binary target labels 0 and 1.")

if not set(np.unique(y_test)).issubset({0.0, 1.0}):
    raise ValueError("Test target must contain only 0 and 1.")

In [6]:
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [7]:


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))



def train_logistic_regression(X, y, learning_rate=0.05,
                              epochs=5000):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)
    bias = 0.0

    for epoch in range(epochs):
        linear_output = np.dot(X, weights) + bias
        predictions = sigmoid(linear_output)

        epsilon = 1e-12
        loss = -np.mean(
            y * np.log(predictions + epsilon)
            + (1 - y) * np.log(1 - predictions + epsilon)
        )

        error = predictions - y

        dw = np.dot(X.T, error) / n_samples
        db = np.sum(error) / n_samples

        weights -= learning_rate * dw
        bias -= learning_rate * db

        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return weights, bias


weights, bias = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.05,
    epochs=5000
)

Epoch 0, Loss: 0.6931
Epoch 500, Loss: 0.3558
Epoch 1000, Loss: 0.3543
Epoch 1500, Loss: 0.3542
Epoch 2000, Loss: 0.3541
Epoch 2500, Loss: 0.3541
Epoch 3000, Loss: 0.3541
Epoch 3500, Loss: 0.3541
Epoch 4000, Loss: 0.3541
Epoch 4500, Loss: 0.3541


In [8]:
def predict(X, weights, bias, threshold=0.5):
    probabilities = sigmoid(np.dot(X, weights) + bias)
    predictions = (probabilities >= threshold).astype(int)
    return predictions


y_pred = predict(X_test, weights, bias)

In [9]:
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

accuracy = (TP + TN) / len(y_test)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1_score = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

In [ ]:
print("\nConfusion Matrix Values")
print("True Positives :", TP)
print("True Negatives :", TN)
print("False Positives:", FP)
print("False Negatives:", FN)

print("\nEvaluation Results")
print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1_score:.4f}")

print("\nTraining samples:", len(y_train))
print("Testing samples :", len(y_test))

# Display learned coefficients
print("\nLearned Weights")
for name, weight in zip(
    train_df.drop(columns=["target"]).columns, weights
):
    print(f"{name}: {weight:.4f}")

print(f"Bias: {bias:.4f}")
